(multiple-formula-comparison)=
# Multiple Formula Model Comparison

### model_utils.<br>**run_multiple_formula_models**

```python
run_multiple_formula_models(create_model_func, y_val_columns, df_train, formulas, 
                            x_vals=['age', 'sex', 'site'], main_folder=None, save_log=True, 
                            num_cores=None, force_overwrite=False, save_models=False, 
                            use_multiprocessing=True, formula_names=None, eval_data=None, 
                            affected=None, func_name=None)
```

&nbsp;&nbsp;&nbsp;&nbsp;Compares multiple GAMLSS model formulas across different y-values with optional model evaluation. Supports parallel processing for efficiency and caches results to avoid recomputation.

&nbsp;&nbsp;&nbsp;&nbsp;<span style="background-color: #f5f5f5; padding: 4px 8px; display: inline-block; width: 90%;">**Parameters:**</span>

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**create_model_func** : ***callable*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Function to create a GAMLSS model

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**y_val_columns** : ***list of str*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;List of y-value column names to model

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**df_train** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Training dataframe

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**formulas** : ***list of dict*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;List of formula dictionaries (each with 'mu_formula', 'sigma_formula', 'nu_formula', 'tau_formula')

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**x_vals** : ***list of str***, *default=['age', 'sex', 'site']* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;List of predictor variables

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**main_folder** : ***str***, *default='results'* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Folder for saving results and logs

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**save_log** : ***bool***, *default=True* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Whether to save logs

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**num_cores** : ***int***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Number of CPU cores for multiprocessing (uses 4 if None)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**force_overwrite** : ***bool***, *default=False* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Whether to overwrite existing cached results

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**save_models** : ***bool***, *default=False* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Whether to save fitted models

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**use_multiprocessing** : ***bool***, *default=True* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Whether to use multiprocessing

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**formula_names** : ***list of str***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Names for the formulas (defaults to "formula1", "formula2", etc.)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**eval_data** : ***pandas.DataFrame***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;DataFrame for evaluation

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**affected** : ***array-like***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Array indicating affected status for evaluation. If None, assumes all subjects are healthy

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**func_name** : ***str***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Distribution family name for evaluation (auto-detected if None)

&nbsp;&nbsp;&nbsp;&nbsp;<span style="background-color: #f5f5f5; padding: 4px 8px; display: inline-block; width: 90%;">**Returns:**</span>

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**comparison_df** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;AIC and BIC values for each formula and y-value

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**convergence_df** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Convergence information for each formula and y-value

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**summary_df** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Best models by AIC and BIC for each y-value

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**eval_results** : ***dict***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Evaluation metrics (returned if eval_data provided)

<br>


**Examples:**

In [38]:
import multiprocessing as mp
import os

from normgamlss.gamlss_main import Gamlss
from normgamlss.model_utils import run_multiple_formula_models

# This function gives us the flexibility on what we want to change and what we want to keep constant
# For example, we can keep the nu and tau formula fixed by removing it from this creat_model_func and in the next cell we can
# just change the other parameters or even change the way we want to input

def create_model_func(model_name, x_vals, y_val, mu_formula, sigma_formula, nu_formula, tau_formula):
    def model_func(data):
        gamlss_model = Gamlss(model_name, x_vals, y_val)
        gamlss_model.fit(
            r_code=f"""
            model <- gamlss(
                y ~ {mu_formula},
                sigma.formula = ~ {sigma_formula},
                nu.formula = ~ {nu_formula},
                tau.formula = ~ {tau_formula},
                family = SHASH(),
                method = RS(10)
            )
            """,
            data=data
        )
        return gamlss_model
    return model_func


In [39]:
train_data1.columns

Index(['site', 'sex', 'age', 'feature_volume1', 'feature_volume2',
       'feature_volume3', 'feature_volume4', 'feature_volume5', 'y_thickness1',
       'y_thickness2', 'y_thickness3', 'y_thickness4'],
      dtype='object')

In [40]:
#### Preparing the input
main_folder = r"../../results/formula_comparison"
num_cores = mp.cpu_count()  # Get the number of CPU cores
y_val_columns = ['y_thickness1','y_thickness2', 'y_thickness3', 'y_thickness4']

# Formula's for baseline and target
formula1 = {'mu_formula': 'pvc(age, by = as.factor(sex)) + re(random=~1|site)',
                'sigma_formula': 'pvc(age) + re(random=~1|site)',
                'nu_formula': '1',
                'tau_formula': '1'}

formula2 = {'mu_formula': 'pvc(age, by = as.factor(sex)) + re(random=~1|site) + feature_volume1',
                'sigma_formula': 'pvc(age) + re(random=~1|site) + feature_volume1 + feature_volume2',
                'nu_formula': '1',
                'tau_formula': '1'}

formula3 = {'mu_formula': 'age + re(random=~1|site)',
                'sigma_formula': '1',
                'nu_formula': '1',
                'tau_formula': '1'}

formula4 = {'mu_formula': 'age',
                'sigma_formula': '1',
                'nu_formula': '1',
                'tau_formula': '1'}

print(f"Number of CPU cores: {num_cores}")


Number of CPU cores: 28


In [41]:
comparison_df, convergence_df, summary_df, eval_results = run_multiple_formula_models(
    create_model_func=create_model_func,
    y_val_columns=y_val_columns,
    df_train=train_data1,
    eval_data=test_data1,
    affected = test_data1['affected'],
    formulas=[formula1, formula2, formula3, formula4],
    x_vals=['age', 'sex', 'site', 'feature_volume1', 'feature_volume2'],
    main_folder=main_folder,
    save_log=True,
    num_cores=4,
    force_overwrite=True,
    save_models=True,
    use_multiprocessing=True
)

2026-10-09 13:24:11,283 - INFO - Using multiprocessing for processing
2026-10-09 13:24:11,284 - INFO - Using 4 cores for multiprocessing
2026-10-09 13:24:11,290 - INFO - Detected distribution family: SHASH


  0%|          | 0/4 [00:00<?, ?it/s]

2026-10-09 13:24:11,378 - INFO - Processing y_thickness2
2026-10-09 13:24:11,378 - INFO - Processing y_thickness1
2026-10-09 13:24:11,380 - INFO - Processing y_thickness4
2026-10-09 13:24:11,380 - INFO - Processing y_thickness3
GAMLSS-RS iteration 1: Global Deviance = 4164.66 
GAMLSS-RS iteration 1: Global Deviance = 3594.915 
GAMLSS-RS iteration 2: Global Deviance = 4119.447 
GAMLSS-RS iteration 1: Global Deviance = 3517.461 
GAMLSS-RS iteration 1: Global Deviance = 3517.72 
GAMLSS-RS iteration 2: Global Deviance = 3509.95 
GAMLSS-RS iteration 3: Global Deviance = 4085.34 
GAMLSS-RS iteration 4: Global Deviance = 4057.893 
GAMLSS-RS iteration 5: Global Deviance = 4037.394 
GAMLSS-RS iteration 3: Global Deviance = 3446.16 
GAMLSS-RS iteration 2: Global Deviance = 3438.841 
GAMLSS-RS iteration 2: Global Deviance = 3451.642 
GAMLSS-RS iteration 6: Global Deviance = 4021.056 
GAMLSS-RS iteration 7: Global Deviance = 4007.78 
GAMLSS-RS iteration 8: Global Deviance = 3997.277 
GAMLSS-RS ite

 25%|██▌       | 1/4 [01:42<05:06, 102.23s/it]

GAMLSS-RS iteration 4: Global Deviance = 3343.595 
GAMLSS-RS iteration 5: Global Deviance = 3297.542 
GAMLSS-RS iteration 6: Global Deviance = 3250.043 
GAMLSS-RS iteration 7: Global Deviance = 3194.299 
GAMLSS-RS iteration 8: Global Deviance = 3128.427 
GAMLSS-RS iteration 9: Global Deviance = 3053.878 
GAMLSS-RS iteration 10: Global Deviance = 2979.143 
Model saved to ../../results/formula_comparison/model_cache/saved_models/model_y_thickness2_formula2.rds
Using 370 healthy subjects out of 470 total subjects for evaluation
Using 370 healthy subjects out of 470 total subjects for evaluation
2026-10-09 13:26:25,515 - INFO - Successfully evaluated formula2 for y_thickness2
GAMLSS-RS iteration 1: Global Deviance = 3594.557 
GAMLSS-RS iteration 2: Global Deviance = 3515.163 
GAMLSS-RS iteration 3: Global Deviance = 3458.398 
GAMLSS-RS iteration 4: Global Deviance = 3416.46 
GAMLSS-RS iteration 5: Global Deviance = 3384.453 
GAMLSS-RS iteration 6: Global Deviance = 3359.281 
GAMLSS-RS iter

100%|██████████| 4/4 [02:36<00:00, 39.10s/it]

2026-10-09 13:26:47,875 - INFO - Processing evaluation results...
2026-10-09 13:26:47,912 - INFO - Evaluation results saved to ../../results/formula_comparison/evaluation_results


In [42]:
comparison_df

,y_val,formula1_AIC,formula1_BIC,formula2_AIC,formula2_BIC,formula3_AIC,formula3_BIC,formula4_AIC,formula4_BIC,best_formula_AIC,best_formula_BIC
0,y_thickness1,3225.840790,3282.887611,-899.242764,-790.030204,3316.726694,3336.475953,3318.726693,3343.413264,formula2 (-899.24),formula2 (-790.03)
1,y_thickness2,3257.353180,3330.216783,3017.498921,3112.186685,3304.799906,3328.668887,3305.812026,3330.498596,formula2 (3017.50),formula2 (3112.19)
2,y_thickness3,4002.630989,4054.352387,4005.850849,4072.592315,4024.239885,4043.989147,4026.239885,4050.926455,formula1 (4002.63),formula3 (4043.99)
3,y_thickness4,3268.824048,3316.117143,3274.029325,3336.467222,3319.858069,3339.607326,3321.858069,3346.544639,formula1 (3268.82),formula1 (3316.12)


In [43]:
summary_df

,y_val,best_AIC_formula,best_AIC_value,best_BIC_formula,best_BIC_value
0,y_thickness1,formula2,-899.242764,formula2,-790.030204
1,y_thickness2,formula2,3017.498921,formula2,3112.186685
2,y_thickness3,formula1,4002.630989,formula3,4043.989147
3,y_thickness4,formula1,3268.824048,formula1,3316.117143


In [44]:
convergence_df

,y_val,formula1_convergence,formula2_convergence,formula3_convergence,formula4_convergence
0,y_thickness1,False,False,False,False
1,y_thickness2,False,False,False,False
2,y_thickness3,False,False,False,False
3,y_thickness4,False,False,False,False


In [45]:
eval_results.keys()

dict_keys(['y_thickness1', 'y_thickness2', 'y_thickness3', 'y_thickness4'])

In [46]:
eval_results['y_thickness1']

,Category,formula1,formula2,formula3,formula4
Metric,,,,,
Z_Scores_Mean,GAMLSS,0.036957,0.045969,0.066339,0.066351
Z_Scores_SD,GAMLSS,1.022716,1.002222,1.003759,1.003759
Logarithmic_Score,GAMLSS,-1.519019,0.460319,-1.556559,-1.556560
Skewness,GAMLSS,-0.066628,0.112544,-0.014668,-0.014668
Kurtosis,GAMLSS,-0.987648,-0.629865,-0.758236,-0.758236
Normality_Test_Statistic,GAMLSS,0.974247,0.989953,0.986737,0.986737
MAE,ML,0.943473,0.119516,0.962069,0.962069
MSE,ML,1.228213,0.022085,1.324287,1.324289
RMSE,ML,1.108248,0.148611,1.150777,1.150778


### Loading Models from Batch Operations

When using batch operations like `run_multiple_formula_models`, `model_comparison`, or `run_models_with_permutation`, models are automatically saved in organized directory structures.

**Model Naming Convention:**
- **Formula comparison**: `"model_{y_val}_{formula_name}.rds"`
- **Base/Target comparison**: `"model_{y_val}_{model_type}.rds"` where model_type is 'base' or 'target'
- **Permutation testing**: `"model_{y_val}_{model_type}_perm_{seed}.rds"` for permuted models

**Note**: The models are saved in `{main_folder}/model_cache/saved_models/`

**Examples:**

In [47]:
## Example code to load a saved model

# Update these values to match your setup
main_folder = r"../../results/formula_comparison"  # Path to your main results folder
y_val = "y_thickness2"  # Replace with your outcome variable
formula_name = "formula1"  # String for the formula number

# The rest is derived automatically from the values above
model_name = f"model_{y_val}_{formula_name}"
model_path = os.path.join(main_folder, "model_cache", "saved_models", f"{model_name}.rds")
loaded_model = Gamlss.load_model(model_path)

Model 'model_y_thickness2_formula1' loaded from ../../results/formula_comparison/model_cache/saved_models/model_y_thickness2_formula1.rds


In [48]:
# The loaded model can now be used like any other -- for example, to calculate z-scores
z_scores = loaded_model.z_score(train_data1,train_data1, func_name = 'SHASH')
print('Z-score:', z_scores)

Z-score: [ 0.21042904 -1.13933524  1.41284508 ... -0.810563   -0.1629004
  1.49969811]
